# MIDI composer: fine-tune a small LLM to write classical piano music from a brief

**Before running in Colab**
1. Runtime > Change runtime type > **GPU** (T4 works; A100/L4 is much faster).
2. Put your MIDI files in Google Drive at `MyDrive/MIDI/` (or upload a zip when the MIDI cell asks).
3. Optional but needed for real training: add a Colab Secret (key icon in the left sidebar) named `GEMINI_API_KEY`. Without it, descriptions fall back to metadata-only one-liners, which are fine for a smoke test only.

Leave `SMOKE_TEST = True` for the first run (12 pieces, 30 training steps, short generation) to check that every stage works, then set it to `False`.

Everything slow (encoded text, descriptions, checkpoints, outputs) is cached in `MyDrive/midi-composer/`, so after a disconnect you can rerun from the top and it picks up where it left off.

In [ ]:
import glob, json, os, sys

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/19katz/midi-composer.git'

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_DIR = '/content/midi-composer'
    if os.path.exists(REPO_DIR):
        !git -C {REPO_DIR} pull -q
    else:
        !git clone -q {REPO_URL} {REPO_DIR}
    !pip install -q pretty_midi mido google-genai datasets
    !apt-get -qq install -y fluidsynth fluid-soundfont-gm > /dev/null
else:
    REPO_DIR = os.path.abspath('..')

sys.path.insert(0, os.path.join(REPO_DIR, 'src'))

In [ ]:
SMOKE_TEST = True

if IN_COLAB:
    WORK_DIR = '/content/drive/MyDrive/midi-composer'
    MIDI_DIR = '/content/drive/MyDrive/MIDI'
else:
    WORK_DIR = os.path.join(REPO_DIR, 'data')
    MIDI_DIR = os.path.expanduser('~/MIDI')
RUN = 'smoke' if SMOKE_TEST else 'full'
TEXT_DIR = f'{WORK_DIR}/text'
AUDIO_DIR = f'{WORK_DIR}/audio'
CKPT_DIR = f'{WORK_DIR}/checkpoints/{RUN}'
ADAPTER_DIR = f'{WORK_DIR}/adapters/{RUN}'
OUT_DIR = f'{WORK_DIR}/outputs/{RUN}'
for d in (TEXT_DIR, AUDIO_DIR, CKPT_DIR, OUT_DIR):
    os.makedirs(d, exist_ok=True)

DESCRIBE_MODEL = 'gemini-2.5-pro'       # any Gemini model that accepts audio
BASE_MODEL = 'unsloth/Qwen2.5-3B-Instruct'
MAX_LEN = 5120                          # longest training window is ~4.7k tokens
TRANSPOSITIONS = range(-3, 4)           # semitone shifts to sample from
PER_PIECE = 3                           # original key + 2 random shifts per piece
EVAL_SEEDS = [0]                        # one generated piece per test work per seed
EVAL_LIMIT = None                       # None = every held-out work; or an int to cap it

## 1. MIDI files

In [ ]:
if IN_COLAB and not glob.glob(f'{MIDI_DIR}/*.mid'):
    from google.colab import files
    print(f'No MIDI files in {MIDI_DIR}. Upload a .zip of them:')
    os.makedirs(MIDI_DIR, exist_ok=True)
    for name in files.upload():
        !unzip -qoj "{name}" '*.mid' -d "{MIDI_DIR}"

paths = sorted(glob.glob(f'{MIDI_DIR}/*.mid'))
if SMOKE_TEST:
    paths = paths[::max(1, len(paths) // 12)][:12]
names = [os.path.basename(p) for p in paths]
print(len(paths), 'pieces')

## 2. Encode MIDI to text

Format is documented in `src/midicomposer/codec.py`.

In [ ]:
import warnings
import pretty_midi
from midicomposer.codec import encode

warnings.filterwarnings('ignore')
texts = {}
for p, n in zip(paths, names):
    cache = f'{TEXT_DIR}/{n[:-4]}.txt'
    if not os.path.exists(cache):
        with open(cache, 'w') as fh:
            fh.write(encode(pretty_midi.PrettyMIDI(p)))
    texts[n] = open(cache).read()

print(texts[names[0]][:800])

## 3. Composition-brief descriptions

Each piece is rendered to audio and sent to Gemini with its metadata and per-bar analysis. One JSON line per piece is appended to Drive as it goes, so this resumes after a disconnect. The full collection is about 23 hours of audio.

In [ ]:
from midicomposer.describe import describe_all

try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY')

if GEMINI_API_KEY:
    from google import genai
    DESC_PATH = f'{WORK_DIR}/descriptions.jsonl'
    describe_all(paths, DESC_PATH, genai.Client(api_key=GEMINI_API_KEY), DESCRIBE_MODEL, AUDIO_DIR)
else:
    print('No GEMINI_API_KEY: using metadata-only descriptions (smoke test only).')
    DESC_PATH = f'{WORK_DIR}/descriptions_fallback.jsonl'
    describe_all(paths, DESC_PATH)

descs = {}
for line in open(DESC_PATH):
    d = json.loads(line)
    descs[d['file']] = d
print(f'{sum(n in descs for n in names)}/{len(names)} described')
print(json.dumps(descs[names[0]], indent=1)[:1500])

## 4. Training windows

Each piece is cut into windows of up to 32 bars. The first window asks for the header (key, meter, length, section plan) and the opening bars; later windows give the header and recent bars and ask for the next ones. Train and test are split by **work**, so every movement of a sonata lands on the same side.

In [ ]:
from midicomposer.dataset import build_split, split_by_work

train_files, test_files = split_by_work(names, test_frac=0.1)
train_examples = build_split(texts, descs, train_files, TRANSPOSITIONS, per_piece=PER_PIECE)
test_examples = build_split(texts, descs, test_files)
json.dump({'train': train_files, 'test': test_files}, open(f'{WORK_DIR}/split_{RUN}.json', 'w'))

approx = sum(len(m['content']) for e in train_examples for m in e['prompt'] + e['completion']) / 1.25
print(f'{len(train_files)} train / {len(test_files)} test pieces')
print(f'{len(train_examples)} training windows, ~{approx / 1e6:.1f}M tokens per epoch')
print(train_examples[0]['prompt'][1]['content'][:700])

## 5. LoRA fine-tune (GPU)

If pip asks you to restart the runtime after installing Unsloth, restart and run all cells again; the cached data makes that quick.

Rough throughput for a 3B model with QLoRA: a T4 manages a few hundred thousand tokens per hour; an A100 is roughly 10x faster. A full run (~20M tokens) is a job for an A100.

In [ ]:
%pip install -q unsloth

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only
import torch

print(torch.cuda.get_device_name(0))
model, tokenizer = FastLanguageModel.from_pretrained(BASE_MODEL, max_seq_length=MAX_LEN, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=32, lora_alpha=32, lora_dropout=0, random_state=0,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    use_gradient_checkpointing='unsloth')

In [ ]:
from datasets import Dataset
from trl import SFTConfig, SFTTrainer

train_ds = Dataset.from_list([
    {'text': tokenizer.apply_chat_template(e['prompt'] + e['completion'], tokenize=False)}
    for e in train_examples
])

cfg = dict(
    output_dir=CKPT_DIR, dataset_text_field='text', packing=False,
    per_device_train_batch_size=1, gradient_accumulation_steps=8,
    learning_rate=2e-4, lr_scheduler_type='cosine', warmup_ratio=0.03,
    num_train_epochs=1, max_steps=30 if SMOKE_TEST else -1,
    optim='adamw_8bit', bf16=is_bfloat16_supported(), fp16=not is_bfloat16_supported(),
    logging_steps=5, save_steps=200, save_total_limit=2, report_to='none', seed=0,
)
try:
    args = SFTConfig(max_length=MAX_LEN, **cfg)
except TypeError:
    args = SFTConfig(max_seq_length=MAX_LEN, **cfg)
try:
    trainer = SFTTrainer(model=model, processing_class=tokenizer, train_dataset=train_ds, args=args)
except TypeError:
    trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=train_ds, args=args)

# Loss only on the score the model writes, not on the brief.
trainer = train_on_responses_only(
    trainer, instruction_part='<|im_start|>user\n', response_part='<|im_start|>assistant\n')

resume = bool(glob.glob(f'{CKPT_DIR}/checkpoint-*'))
trainer.train(resume_from_checkpoint=True if resume else None)
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

## 6. Compose and evaluate

Generation runs window by window until the model writes `<|piece_end|>` or reaches the length it planned. On a T4, a 3B model takes a few minutes per 32-bar window, so the full test set (~30 works) is a job for an A100.

For every held-out work and seed, the eval cell saves to `outputs/<run>/<tag>/`:
- `<piece>__<tag>__s<seed>.mid`, `.txt` (score), and `.wav` (render)

and appends a line to `outputs/<run>/results.jsonl` with the brief, key, meter, sampling settings, metrics, the same metrics for the real piece as a baseline, and the file paths. `<tag>` names the base model, LoRA rank, training step, and temperature, so different checkpoints and settings sit side by side. Pieces already logged for a tag and seed are skipped, so rerunning after a disconnect resumes.

To evaluate a saved adapter in a fresh session, run the setup cells, load it with `FastLanguageModel.from_pretrained(ADAPTER_DIR, max_seq_length=MAX_LEN, load_in_4bit=True)` instead of training, and set `STEP` below to name it (e.g. `'step1800'`).

In [ ]:
from IPython.display import Audio, display
from transformers import set_seed
from midicomposer.evaluate import hand_lines
from midicomposer.runs import run_eval

FastLanguageModel.for_inference(model)
GEN = dict(max_new_tokens=3500, temperature=0.9, top_p=0.95)

def generate_fn(messages):
    ids = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors='pt').to(model.device)
    out = model.generate(input_ids=ids, do_sample=True, **GEN)
    return tokenizer.decode(out[0, ids.shape[1]:], skip_special_tokens=True)

STEP = f'step{trainer.state.global_step}' if 'trainer' in globals() else 'adapter'
TAG = f"{BASE_MODEL.split('/')[-1]}-r32-{STEP}-t{GEN['temperature']}"
train_lines = {l for f in train_files for l in hand_lines(texts[f])}
eval_files = test_files[:1] if SMOKE_TEST else test_files[:EVAL_LIMIT]
max_chunks = 2 if SMOKE_TEST else 40

def show(row):
    m, ref = row['metrics'], row['reference_metrics']
    print(f"{row['file']} seed {row['seed']}: {m['bars']}/{m['planned_bars']} bars, "
          f"valid tokens {m['valid_token_frac']:.3f}, key match {m.get('key_match')}, "
          f"copied lines {m['copied_line_frac']:.3f} (real piece: {ref['copied_line_frac']:.3f}), {row['seconds']}s")
    if row['paths']['wav']:
        display(Audio(row['paths']['wav']))

print(f'Tag {TAG}: {len(eval_files)} works x {len(EVAL_SEEDS)} seeds -> {OUT_DIR}/{TAG}/')
rows = run_eval(generate_fn, eval_files, texts, descs, OUT_DIR, TAG, seeds=EVAL_SEEDS,
                max_chunks=max_chunks, train_lines=train_lines, set_seed=set_seed,
                settings=GEN, on_piece=show)

In [ ]:
import pandas as pd
from midicomposer.runs import load_results

df = pd.json_normalize(load_results(f'{OUT_DIR}/results.jsonl'))
cols = ['metrics.valid_token_frac', 'metrics.bars', 'metrics.planned_bars', 'metrics.key_match',
        'metrics.pc_hist_l1_to_reference', 'metrics.copied_line_frac',
        'reference_metrics.copied_line_frac', 'seconds']
display(df.groupby('tag')[cols].mean(numeric_only=False).round(3))
display(df[['tag', 'file', 'seed'] + cols + ['paths.mid']])

In [ ]:
import time
from midicomposer.compose import compose
from midicomposer.evaluate import evaluate
from midicomposer.runs import save_piece

BRIEF = '''Write a melancholy nocturne that opens with a singing right-hand melody over a
rocking left-hand accompaniment, grows more agitated and chromatic in the middle section,
climaxes around two thirds of the way through, and fades to a quiet, consoling close.'''
KEY, METER, SEED = 'C# minor', '4/4', 0

set_seed(SEED)
text = compose(generate_fn, BRIEF, key=KEY, meter=METER, max_chunks=max_chunks,
               on_chunk=lambda n, total: print(f'  {n}/{total or "?"} bars'))
name = f"custom__{TAG}__s{SEED}__{time.strftime('%Y%m%d-%H%M%S')}"
paths = save_piece(text, f'{OUT_DIR}/custom', name)
metrics = evaluate(text, requested_key=KEY, train_lines=train_lines)
json.dump({'brief': BRIEF, 'key': KEY, 'meter': METER, 'seed': SEED, 'tag': TAG, 'settings': GEN,
           'metrics': metrics, 'paths': paths}, open(f'{OUT_DIR}/custom/{name}.json', 'w'), indent=1)
print(metrics)
if paths['wav']:
    display(Audio(paths['wav']))